# Rayleigh calibration — quickstart

Run the molecular (Rayleigh) calibration on a single night of ceilometer / lidar
data with the `calibration` package.

**Prerequisite** — install once from the repository root:

```bash
pip install -e ".[plotting]"
```

You also need input data on disk (E-PROFILE L1/L2 or RAW NetCDF). For 910 nm
instruments a matching-month CAMS file is required for the mandatory water-vapour
correction. Run notebooks from the repo root (or from `examples/`).

## 1. Imports

In [ ]:
from pathlib import Path

from calibration import (
    calibrate_rayleigh,
    CalibrationOptions,
    InstrumentInfo,
)
from calibration.config import InstrumentType, DataLevel

## 2. Load the calibration options

`options.json` (repo root) holds the processing configuration: input data level,
archive root, nighttime window, quality thresholds, molecular-window method and the
water-vapour settings. Load it, then override the paths that point at *your* data.

In [ ]:
# robust whether the notebook is launched from the repo root or from examples/
OPTIONS_PATH = Path("options.json")
if not OPTIONS_PATH.exists():
    OPTIONS_PATH = Path("..") / "options.json"

options = CalibrationOptions.from_json(OPTIONS_PATH)

# --- point these at your own archive -------------------------------------
# options.folder_root  = Path("D:/E-PROFILE_L1")     # input data archive
# options.cams_folder  = Path("D:/CAMS/")            # CAMS monthly means (910 nm WV)
# options.folder_output = Path(r"C:/DATA/Projects/202606_E-PROFILE_calibration/example")

for k in ["data_level", "molecular_method", "apply_wv_correction", "hour_min", "hour_max"]:
    print(f"{k:20s}: {getattr(options, k, '(n/a)')}")

## 3. Describe the instrument

In [ ]:
info = InstrumentInfo(
    site_name="Payerne",
    wmo_id="0-20000-0-06610",
    identifier="A",
    instrument_type=InstrumentType.CHM15k,
    latitude=46.82,
    longitude=6.95,
    altitude=491,
)
info

## 4. Run the calibration for one night

`calibrate_rayleigh(date, instrument, options)` reads that night's data, selects a
clear molecular window, fits the Rayleigh slope and returns a `CalibrationResult`.
It needs the data files under `options.folder_root`, so the cell reports a friendly
message if the data is not present on this machine.

In [ ]:
DATE = "20240115"  # YYYYMMDD

try:
    result = calibrate_rayleigh(DATE, info, options)
    for k in ["flag", "flag_meaning", "lidar_constant", "uncertainty",
              "bottom_height", "top_height", "message"]:
        print(f"{k:16s}: {getattr(result, k, '(n/a)')}")
except FileNotFoundError as e:
    print("No input data found for this date on this machine.")
    print("Point options.folder_root at your archive and pick an available date.")
    print("Details:", e)

## 5. Flag meanings

| flag | meaning |
|---|---|
| 1 | successful calibration |
| 0.5 | partially clear night |
| 0 | no data |
| -1 | not a clear night |
| -2 | signal not proportional to molecular scattering (fog) |
| -3 | poor agreement between methods |
| -4 | missing CAMS water-vapour data (910 nm skipped, no fallback) |
| -5..-8 | NaN RCS / uncertainty too large / bad Rayleigh slope |

## 6. Batch processing & CLI

```bash
# one date, verbose
python -m calibration.main --date 20240115 --verbose

# a date range with 8 parallel workers
rayleigh-calibration --date 20240101 20240131 -j 8
```

For full-archive or site studies see the runnable scripts in `scripts/`
(`run_all_l2monthly.py`, `run_calibration.py`, ...) and the studies in `validation/`.